In [ ]:
this is the final code

In [2]:
# Install required libraries (if not already installed)
!pip install pycox torchtuples scikit-survival matplotlib pandas numpy scikit-learn torch --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.0/300.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.7/142.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.6/413.6 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.4 MB/s eta 0:00:00


In [ ]:
pip install scikit-survival

In [ ]:
 !pip install pycox torchtuples pandas scikit-learn matplotlib numpy --quiet

In [ ]:
pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 14.5 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=c5ab5521ffba6917a6f56be1d9eb43ede9024a695e6b12b30c1cb7348433b820
  Stored in directory: /root/.cache/pip/wheels/8b/67/f4/2caaae2146198dcb824f31a303833b07b14a5ec863fb3acd7b
Successfully built autograd-gamma


In [ ]:
# --- IMPORTS ---
import pandas as pd
import numpy as np
import torch
import gc
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from pycox.models import DeepHitSingle, CoxTime
from pycox.evaluation import EvalSurv
import torchtuples as tt
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv
import warnings
warnings.filterwarnings('ignore')


import pandas as pd
import numpy as np
import gc  # Garbage collection
import torch
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from pycox.models import DeepHitSingle, CoxTime
from pycox.evaluation import EvalSurv
import torchtuples as tt
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
np.random.seed(42)
torch.manual_seed(42)

# --- Memory-optimized data loading and preprocessing ---
print("Loading and preprocessing data...")
data = pd.read_csv("/content/train.csv")
duration_col = 'efs_time'
event_col = 'efs'

# Identify columns
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
numerical_cols = data.select_dtypes(include=['int64', 'float64']).drop(
    columns=[duration_col, event_col], errors='ignore').columns.tolist()

# Memory-optimized transformers
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Use sparse matrices to save memory
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

X = data.drop(columns=[duration_col, event_col])
y = data[[duration_col, event_col]].copy()

# Clean up memory
del data
gc.collect()

# Transform data
print("Transforming data...")
X_processed = preprocessor.fit_transform(X)

# Create structured array for survival analysis
def create_structured_array(event, time):
    return np.array([(event[i], time[i]) for i in range(len(event))],
                    dtype=[('event', 'bool'), ('time', 'float64')])

y_structured = create_structured_array(y[event_col].values, y[duration_col].values)

# Train/Val/Test split
print("Splitting data...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X_processed, y, test_size=0.3, stratify=y[event_col], random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp[event_col], random_state=42
)

# Convert to appropriate formats
if hasattr(X_train, "toarray"):
    X_train_dense = X_train.toarray().astype('float32')
    X_val_dense = X_val.toarray().astype('float32')
    X_test_dense = X_test.toarray().astype('float32')
else:
    X_train_dense = X_train.astype('float32')
    X_val_dense = X_val.astype('float32')
    X_test_dense = X_test.astype('float32')

y_train_structured = create_structured_array(y_train[event_col].values, y_train[duration_col].values)
y_val_structured = create_structured_array(y_val[event_col].values, y_val[duration_col].values)
y_test_structured = create_structured_array(y_test[event_col].values, y_test[duration_col].values)

# Clean up memory
del X, y, X_temp, y_temp
gc.collect()

# --- Helper function: C-index ---
def compute_c_index(model, X, y_structured):
    if hasattr(model, 'predict_surv_df'):
        # For PyCox models
        surv = model.predict_surv_df(X)
        ev = EvalSurv(surv, y_structured['time'], y_structured['event'], censor_surv='km')
        return ev.concordance_td('antolini')
    else:
        # For scikit-survival models
        pred = model.predict(X)
        from sksurv.metrics import concordance_index_censored
        result = concordance_index_censored(
            y_structured['event'],
            y_structured['time'],
            pred
        )
        return result[0]

In [ ]:
# ========== Model 1: DeepHitSingle ==========
print("Training DeepHitSingle...")
num_durations = 120
labtrans = DeepHitSingle.label_transform(num_durations)
y_train_transformed = labtrans.fit_transform(y_train[duration_col].values, y_train[event_col].values)
y_val_transformed = labtrans.transform(y_val[duration_col].values, y_val[event_col].values)
y_test_transformed = labtrans.transform(y_test[duration_col].values, y_test[event_col].values)

train = (X_train, y_train_transformed)
val = (X_val, y_val_transformed)
test = (X_test, y_test_transformed)

# Neural net definition (large capacity)
in_features = X_train.shape[1]
num_nodes = [512, 256, 128]
out_features = labtrans.out_features
net = tt.practical.MLPVanilla(in_features, num_nodes, out_features, activation=torch.nn.ReLU, dropout=0.3)
model_deephit = DeepHitSingle(net, tt.optim.Adam(lr=1e-3), alpha=0.2, sigma=0.1, duration_index=labtrans.cuts, device=str(device))

batch_size = 256
epochs = 200
callbacks = [tt.callbacks.EarlyStopping(patience=20)]
log_deephit = model_deephit.fit(train[0], train[1], batch_size, epochs, callbacks, verbose=True, val_data=val)

surv_deephit = model_deephit.predict_surv_df(test[0])
ev_deephit = EvalSurv(surv_deephit, y_test[duration_col].values, y_test[event_col].values, censor_surv='km')
c_index_deephit_test = ev_deephit.concordance_td('antolini')
surv_deephit_train = model_deephit.predict_surv_df(train[0])
ev_deephit_train = EvalSurv(surv_deephit_train, y_train[duration_col].values, y_train[event_col].values, censor_surv='km')
c_index_deephit_train = ev_deephit_train.concordance_td('antolini')
surv_deephit_val = model_deephit.predict_surv_df(val[0])
ev_deephit_val = EvalSurv(surv_deephit_val, y_val[duration_col].values, y_val[event_col].values, censor_surv='km')
c_index_deephit_val = ev_deephit_val.concordance_td('antolini')
print(f"DeepHit C-index (Train/Val/Test): {c_index_deephit_train:.4f} / {c_index_deephit_val:.4f} / {c_index_deephit_test:.4f}")

Training DeepHitSingle...
0:	[0s / 0s],		train_loss: 0.7992,	val_loss: 0.7116
1:	[0s / 1s],		train_loss: 0.7246,	val_loss: 0.6765
2:	[0s / 1s],		train_loss: 0.6873,	val_loss: 0.6535
3:	[0s / 2s],		train_loss: 0.6608,	val_loss: 0.6380
4:	[0s / 2s],		train_loss: 0.6333,	val_loss: 0.6235
5:	[0s / 3s],		train_loss: 0.6117,	val_loss: 0.6061
6:	[0s / 3s],		train_loss: 0.5882,	val_loss: 0.5937
7:	[0s / 4s],		train_loss: 0.5772,	val_loss: 0.5811
8:	[0s / 4s],		train_loss: 0.5691,	val_loss: 0.5782
9:	[0s / 5s],		train_loss: 0.5565,	val_loss: 0.5780
10:	[0s / 5s],		train_loss: 0.5493,	val_loss: 0.5756
11:	[0s / 6s],		train_loss: 0.5393,	val_loss: 0.5754
12:	[0s / 6s],		train_loss: 0.5358,	val_loss: 0.5798
13:	[0s / 7s],		train_loss: 0.5292,	val_loss: 0.5788
14:	[0s / 7s],		train_loss: 0.5200,	val_loss: 0.5813
15:	[0s / 8s],		train_loss: 0.5156,	val_loss: 0.5924
16:	[0s / 8s],		train_loss: 0.5089,	val_loss: 0.5913
17:	[0s / 9s],		train_loss: 0.5062,	val_loss: 0.6033
18:	[0s / 9s],		train_loss: 0.